In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [2]:
df = pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv').drop(columns=['store room','floor_category','balcony'])

In [3]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,flat,sector 104,4.45,4,5,Relatively New,7222.0,0,1,Medium
1,flat,sector 86,1.26,3,3,Moderately Old,1747.0,1,1,Medium
2,flat,sector 9a,1.50,4,3,Relatively New,1833.0,0,0,Low
3,flat,sector 37d,0.75,2,2,Moderately Old,1410.0,0,1,Medium
4,flat,sector 86,1.45,3,2,Relatively New,1480.0,0,2,Medium


In [4]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished

# Numerical = bedRoom, bathroom, built_up_area, servant room
# Ordinal = property_type, furnishing_type, luxury_category 
# OHE = sector, agePossession

In [5]:
df['agePossession'] = df['agePossession'].replace(
    {
        'Relatively New':'new',
        'Moderately Old':'old',
        'New Property' : 'new',
        'Old Property' : 'old',
        'Under Construction' : 'under construction'
    }
)

In [6]:
df['agePossession'].unique()

array(['new', 'old', 'under construction'], dtype=object)

In [7]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,flat,sector 104,4.45,4,5,new,7222.0,0,1,Medium
1,flat,sector 86,1.26,3,3,old,1747.0,1,1,Medium
2,flat,sector 9a,1.50,4,3,new,1833.0,0,0,Low
3,flat,sector 37d,0.75,2,2,old,1410.0,0,1,Medium
4,flat,sector 86,1.45,3,2,new,1480.0,0,2,Medium


In [8]:
df['property_type'] = df['property_type'].replace({'flat':0,'house':1})

C:\Users\virat\AppData\Local\Temp\ipykernel_22020\71934247.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['property_type'] = df['property_type'].replace({'flat':0,'house':1})


In [9]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,0,sector 104,4.45,4,5,new,7222.0,0,1,Medium
1,0,sector 86,1.26,3,3,old,1747.0,1,1,Medium
2,0,sector 9a,1.50,4,3,new,1833.0,0,0,Low
3,0,sector 37d,0.75,2,2,old,1410.0,0,1,Medium
4,0,sector 86,1.45,3,2,new,1480.0,0,2,Medium


In [10]:
df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})

C:\Users\virat\AppData\Local\Temp\ipykernel_22020\2576796079.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})


In [11]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,0,sector 104,4.45,4,5,new,7222.0,0,1,1
1,0,sector 86,1.26,3,3,old,1747.0,1,1,1
2,0,sector 9a,1.50,4,3,new,1833.0,0,0,0
3,0,sector 37d,0.75,2,2,old,1410.0,0,1,1
4,0,sector 86,1.45,3,2,new,1480.0,0,2,1


In [12]:
new_df = pd.get_dummies(df,columns=['sector','agePossession'],drop_first=True)

In [13]:
X = new_df.drop(columns=['price'])
y = new_df['price']

In [14]:
y_log = np.log1p(y)

In [15]:
y_log

0       1.695616
1       0.815365
2       0.916291
3       0.559616
4       0.896088
          ...   
3549    0.717840
3550    0.875469
3551    0.336472
3552    1.081805
3553    0.792993
Name: price, Length: 3554, dtype: float64

In [16]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [17]:
X_scaled

array([[-0.51717989,  0.72860975,  1.19490906, ..., -0.05571996,
        -0.6074778 , -0.19092872],
       [-0.51717989, -0.07432949, -0.18456367, ..., -0.05571996,
         1.64615069, -0.19092872],
       [-0.51717989,  0.72860975, -0.18456367, ..., -0.05571996,
        -0.6074778 , -0.19092872],
       ...,
       [-0.51717989, -0.87726873, -0.87430004, ..., -0.05571996,
        -0.6074778 , -0.19092872],
       [ 1.9335632 , -0.07432949,  1.19490906, ..., -0.05571996,
         1.64615069, -0.19092872],
       [-0.51717989, -0.07432949, -0.18456367, ..., -0.05571996,
         1.64615069, -0.19092872]], shape=(3554, 121))

In [18]:
X_scaled = pd.DataFrame(X_scaled , columns = X.columns)

In [19]:
X_scaled

,property_type,bedRoom,bathroom,built_up_area,servant room,furnishing_type,luxury_category,sector_gwal pahari,sector_manesar,sector_sector 1,...,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sector 99a,sector_sector 9a,sector_sohna road,sector_sohna road road,agePossession_old,agePossession_under construction
0,-0.517180,0.728610,1.194909,4.407356,-0.747968,0.430655,0.444313,-0.071348,-0.044424,-0.092266,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,-0.607478,-0.190929
1,-0.517180,-0.074329,-0.184564,-0.094154,1.336956,0.430655,0.444313,-0.071348,-0.044424,-0.092266,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,1.646151,-0.190929
2,-0.517180,0.728610,-0.184564,-0.023446,-0.747968,-1.458910,-0.979571,-0.071348,-0.044424,-0.092266,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,18.825515,-0.211375,-0.05572,-0.607478,-0.190929
3,-0.517180,-0.877269,-0.874300,-0.371233,-0.747968,0.430655,0.444313,-0.071348,-0.044424,-0.092266,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,1.646151,-0.190929
4,-0.517180,-0.074329,-0.874300,-0.313680,-0.747968,2.320220,0.444313,-0.071348,-0.044424,-0.092266,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,-0.607478,-0.190929
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3549,-0.517180,-0.074329,0.505173,0.113861,1.336956,-1.458910,0.444313,-0.071348,-0.044424,-0.092266,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,-0.607478,-0.190929
3550,-0.517180,-0.074329,-0.184564,-0.386033,-0.747968,-1.458910,1.868197,-0.071348,-0.044424,-0.092266,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,-0.607478,-0.190929
3551,-0.517180,-0.877269,-0.874300,-0.935258,-0.747968,2.320220,-0.979571,-0.071348,-0.044424,-0.092266,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,-0.607478,-0.190929
3552,1.933563,-0.074329,1.194909,-0.774108,-0.747968,0.430655,-0.979571,-0.071348,-0.044424,-0.092266,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,1.646151,-0.190929


In [20]:
X_scaled.columns.to_list()

['property_type',
 'bedRoom',
 'bathroom',
 'built_up_area',
 'servant room',
 'furnishing_type',
 'luxury_category',
 'sector_gwal pahari',
 'sector_manesar',
 'sector_sector 1',
 'sector_sector 102',
 'sector_sector 103',
 'sector_sector 104',
 'sector_sector 105',
 'sector_sector 106',
 'sector_sector 107',
 'sector_sector 108',
 'sector_sector 109',
 'sector_sector 10a',
 'sector_sector 11',
 'sector_sector 110',
 'sector_sector 111',
 'sector_sector 112',
 'sector_sector 113',
 'sector_sector 12',
 'sector_sector 13',
 'sector_sector 14',
 'sector_sector 15',
 'sector_sector 17',
 'sector_sector 17a',
 'sector_sector 17b',
 'sector_sector 2',
 'sector_sector 21',
 'sector_sector 22',
 'sector_sector 23',
 'sector_sector 24',
 'sector_sector 25',
 'sector_sector 26',
 'sector_sector 27',
 'sector_sector 28',
 'sector_sector 3',
 'sector_sector 3 phase 2',
 'sector_sector 3 phase 3 extension',
 'sector_sector 30',
 'sector_sector 31',
 'sector_sector 33',
 'sector_sector 36',
 'sect

In [21]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X_scaled, y_log, cv=kfold, scoring='r2')

In [22]:
scores.mean() , scores.std()

(np.float64(0.8530446592832444), np.float64(0.018318498672489324))

In [23]:
lr = LinearRegression()
ridge = Ridge(alpha=0.0001)

In [24]:
lr.fit(X_scaled,y_log)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [25]:
ridge.fit(X_scaled,y_log)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",Non

In [27]:
coef_df = pd.DataFrame(ridge.coef_.reshape(1,121),columns=X.columns).stack().reset_index().drop(columns=['level_0']).rename(columns={'level_1':'feature',0:'coef'})

In [28]:
coef_df

,feature,coef
0,property_type,0.120979
1,bedRoom,0.054727
2,bathroom,0.066372
3,built_up_area,0.208668
4,servant room,0.049797
...,...,...
116,sector_sector 9a,-0.006014
117,sector_sohna road,-0.031702
118,sector_sohna road road,-0.010527
119,agePossession_old,-0.007731


In [29]:
# 1. Import necessary libraries
import statsmodels.api as sm

# 2. Add a constant to X
X_with_const = sm.add_constant(X_scaled)

# 3. Fit the model
model = sm.OLS(y_log, X_with_const).fit()

# 4. Obtain summary statistics
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.865
Model:                            OLS   Adj. R-squared:                  0.861
Method:                 Least Squares   F-statistic:                     182.1
Date:                Mon, 06 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:42:31   Log-Likelihood:                 592.99
No. Observations:                3554   AIC:                            -942.0
Df Residuals:                    3432   BIC:                            -188.5
Df Model:                         121                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

In [30]:
y_log.std()

np.float64(0.5579613263072811)

In [31]:
X_scaled['bedRoom'].std()

np.float64(1.000140716246387)